# Fine-Tuning Comparison: Popular Model Types

This notebook compares fine-tuning across **five representative model families**:

| Family | Model | Params | Technique |
|--------|-------|--------|-----------|
| GPT | `EleutherAI/pythia-410m` | 405M | Full FT |
| GPT | `gpt2-medium` | 355M | Full FT |
| Llama | `meta-llama/Llama-3.1-8B-Instruct` | 8B | QLoRA |
| Qwen | `Qwen/Qwen3-8B` | 8B | QLoRA |
| MoE | `microsoft/Phi-3.5-MoE-instruct` | 42B (6.6B active) | LoRA |
| SLM | `microsoft/Phi-3.5-mini-instruct` | 3.8B | QLoRA |

**Prerequisites:** Run `python scripts/download_data.py` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import pandas as pd
from pathlib import Path

from src.data.loader import load_stacksample
from src.data.preprocessing import preprocess_qa_pairs, split_dataset
from src.data.tokenization import build_dataset_dict
from src.models.loader import load_tokenizer, load_model
from src.models.quantization import build_quant_config
from src.models.peft_config import apply_peft
from src.training.trainer import run_training, print_training_results
from src.utils.config import load_config

In [ ]:
# ── Load and prepare data once ──────────────────────────────────
questions, answers = load_stacksample(
    data_dir="../data",
    score_threshold=5,
    max_questions=100,
)
qa_pairs = preprocess_qa_pairs(questions, answers)
train_df, test_df, val_df = split_dataset(qa_pairs)

print(f"Train: {train_df.height:,} | Val: {val_df.height:,} | Test: {test_df.height:,}")

In [ ]:
MODEL_CONFIGS = [
    "../configs/pythia_410m.yaml",
    "../configs/gpt2_medium.yaml",
    "../configs/llama_3_1_8b.yaml",
    "../configs/qwen3_8b.yaml",
    "../configs/phi_3_5_moe.yaml",
    "../configs/phi_3_5_mini.yaml",
]

all_results = {}

for config_path in MODEL_CONFIGS:
    cfg = load_config(config_path)
    model_name = cfg["model"]["name"]
    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    # 1. Tokenizer
    tokenizer = load_tokenizer(
        model_name,
        trust_remote_code=cfg["model"].get("trust_remote_code", False),
    )

    # 2. Dataset
    dataset_dict = build_dataset_dict(
        train_df, val_df, tokenizer, model_name,
        max_length=cfg["data"]["max_length"],
    )

    # 3. Model
    quant_config = build_quant_config(cfg.get("quantization"))
    model = load_model(
        model_name,
        quantization_config=quant_config,
        trust_remote_code=cfg["model"].get("trust_remote_code", False),
        attn_implementation=cfg["model"].get("attn_implementation"),
    )

    # 4. PEFT (optional)
    if cfg["model"].get("use_peft", False):
        model = apply_peft(model, cfg["peft"])
        model.print_trainable_parameters()

    # 5. Train
    trainer, monitor, train_result = run_training(
        model, tokenizer, dataset_dict, cfg
    )
    print_training_results(trainer, monitor, train_result)

    all_results[model_name] = {
        "train_result": train_result,
        "epoch_data": monitor.epoch_data,
        "config": cfg,
    }

    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()

print("\n✓ All models fine-tuned.")

In [ ]:
# ── Summary comparison table ─────────────────────────────────────
rows = []
for name, res in all_results.items():
    ep = res["epoch_data"]
    total_time = sum(e["time"] for e in ep)
    avg_cpu = sum(e["avg_cpu"] for e in ep) / len(ep) if ep else 0
    avg_mem = sum(e["avg_memory"] for e in ep) / len(ep) if ep else 0
    final_loss = res["train_result"].metrics.get("train_loss", None)
    rows.append({
        "Model": name.split("/")[-1],
        "Total Time (s)": f"{total_time:.1f}",
        "Avg CPU (%)": f"{avg_cpu:.1f}",
        "Avg Mem (%)": f"{avg_mem:.1f}",
        "Final Loss": f"{final_loss:.4f}" if final_loss else "N/A",
        "Technique": "QLoRA" if "qlora" in res["config"]["model"]["output_dir"] else "Full/LoRA",
    })

df = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("FINE-TUNING COMPARISON SUMMARY")
print("=" * 70)
display(df)